# Phase 5 - Notebook 04: DPT Head Multi-scale Fusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase5/04_dpt_multiscale.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why multi-scale features are essential for dense prediction
2. Implement DPT reassemble operations to reshape ViT tokens back to 2D
3. Build a multi-scale fusion module with progressive refinement
4. Understand different activation functions for depth vs point predictions
5. Compare depth unprojection vs direct point coordinate prediction
6. Use confidence maps to filter unreliable predictions

**Estimated Time**: 60 minutes

**Prerequisites**: Phase 5 Notebooks 01-03 (ViT, DINO, DPT basics)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

## 1. Why Multi-scale Features?

### The Multi-scale Hierarchy in Vision Transformers

Different transformer layers capture different types of information:

| Layer Type | Example Layers | Information | Receptive Field |
|------------|----------------|-------------|------------------|
| **Shallow** | 0-6 | Local edges, textures | Small (1-2 patches) |
| **Mid** | 7-15 | Object parts, structures | Medium (3-8 patches) |
| **Deep** | 16-23 | Global semantics, context | Large (whole image) |

**Why we need all of them for dense prediction:**
- **Deep features alone**: blurry boundaries (lost fine details)
- **Shallow features alone**: no global context (inconsistent predictions)
- **Multi-scale fusion**: best of both worlds!

### DPT's Multi-scale Selection

For a 24-layer ViT, DPT typically uses 4 layers:
```
Layer 4  (shallow)  →  256 channels  ─┐
Layer 11 (mid-low)  →  512 channels  ├─► Fusion
Layer 17 (mid-high) → 1024 channels  │
Layer 23 (deep)     → 1024 channels  ┘
```

In [ ]:
# Create synthetic feature maps at different scales to visualize

def create_synthetic_features(size=224, patch_size=16):
    """
    Create synthetic features that simulate what different layers see.
    """
    H, W = size, size
    Hp, Wp = H // patch_size, W // patch_size
    
    # Create coordinate grids
    x = np.linspace(-1, 1, Hp)
    y = np.linspace(-1, 1, Wp)
    X, Y = np.meshgrid(x, y)
    
    features = {}
    
    # Layer 4 (shallow): high-frequency textures
    layer4 = np.sin(X * 10) * np.cos(Y * 10) + np.random.randn(Hp, Wp) * 0.3
    features['layer4'] = layer4
    
    # Layer 11 (mid): medium-scale structures
    layer11 = np.sin(X * 4) * np.cos(Y * 4) + 0.5 * np.exp(-(X**2 + Y**2) / 0.5)
    features['layer11'] = layer11
    
    # Layer 17 (mid-high): larger structures
    layer17 = np.exp(-(X**2 + Y**2) / 0.8) + 0.3 * np.sin(X * 2) * np.cos(Y * 2)
    features['layer17'] = layer17
    
    # Layer 23 (deep): global patterns
    layer23 = np.exp(-((X-0.3)**2 + (Y+0.2)**2) / 1.2) + 0.5
    features['layer23'] = layer23
    
    return features


# Visualize multi-scale features
features = create_synthetic_features()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
layer_names = ['layer4', 'layer11', 'layer17', 'layer23']
titles = [
    'Layer 4 (Shallow)\nHigh-frequency textures & edges',
    'Layer 11 (Mid-low)\nObject parts & local structures',
    'Layer 17 (Mid-high)\nLarger structures & regions',
    'Layer 23 (Deep)\nGlobal semantics & context'
]
channels = [256, 512, 1024, 1024]

for idx, (ax, layer_name, title, ch) in enumerate(zip(axes.flat, layer_names, titles, channels)):
    im = ax.imshow(features[layer_name], cmap='viridis', interpolation='bilinear')
    ax.set_title(f'{title}\nOutput channels: {ch}', fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Multi-scale Features from Different ViT Layers', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nKey Insight: Different layers capture different scales of information!")
print("- Shallow: Local details (but no context)")
print("- Deep: Global understanding (but blurry)")
print("- Solution: Fuse them together progressively!")

## 2. DPT Reassemble Operation

### The Problem: ViT outputs 1D token sequences

After processing through ViT layers:
```
Input image: [B, 3, 224, 224]
After patchify: [B, 14×14=196 patches, 768 features]
After ViT layer: [B, 196, 768] (still 1D!)
```

But we need 2D spatial structure for:
- Convolutional operations
- Upsampling
- Dense predictions (depth/normals/etc.)

### The Solution: Reassemble operation

**Step 1**: Reshape tokens back to 2D
```
[B, P, C] → [B, H_p, W_p, C]
where P = H_p × W_p (number of patches)
```

**Step 2**: Project to target channels
```
[B, H_p, W_p, C] → Conv1x1 → [B, H_p, W_p, C_out]
```

**Step 3**: Permute to PyTorch format
```
[B, H_p, W_p, C_out] → [B, C_out, H_p, W_p]
```

In [ ]:
class Reassemble(nn.Module):
    """
    Reassemble ViT tokens back to 2D spatial grid.
    
    Converts [B*S, P, C] → [B*S, C_out, H_p, W_p]
    where S = number of source images
    """
    def __init__(self, in_channels, out_channels, patch_size=16, image_size=224):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.patch_size = patch_size
        self.image_size = image_size
        
        # Calculate patch grid dimensions
        self.Hp = image_size // patch_size
        self.Wp = image_size // patch_size
        
        # Projection layer to target channels
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        
    def forward(self, tokens):
        """
        Args:
            tokens: [B*S, P, C] where P = Hp * Wp
        Returns:
            features: [B*S, C_out, Hp, Wp]
        """
        BS, P, C = tokens.shape
        assert P == self.Hp * self.Wp, f"Expected {self.Hp * self.Wp} patches, got {P}"
        assert C == self.in_channels, f"Expected {self.in_channels} channels, got {C}"
        
        # Step 1: Reshape to 2D spatial grid
        # [B*S, P, C] → [B*S, Hp, Wp, C]
        features = tokens.reshape(BS, self.Hp, self.Wp, C)
        
        # Step 2: Permute to [B*S, C, Hp, Wp] for Conv2d
        features = features.permute(0, 3, 1, 2)
        
        # Step 3: Project to target channels
        features = self.projection(features)  # [B*S, C_out, Hp, Wp]
        
        return features


# Test reassemble with different layer configurations
print("Testing Reassemble Operations:\n")
print("="*60)

BS = 2  # Batch size * num source images
image_size = 224
patch_size = 16
P = (image_size // patch_size) ** 2  # 14*14 = 196

# Typical DPT configuration
configs = [
    ('Layer 4',  768,  256),
    ('Layer 11', 768,  512),
    ('Layer 17', 768, 1024),
    ('Layer 23', 768, 1024),
]

for layer_name, in_ch, out_ch in configs:
    # Create synthetic tokens
    tokens = torch.randn(BS, P, in_ch)
    
    # Apply reassemble
    reassemble = Reassemble(in_ch, out_ch, patch_size, image_size)
    features = reassemble(tokens)
    
    print(f"{layer_name}:")
    print(f"  Input:  {list(tokens.shape)} (1D token sequence)")
    print(f"  Output: {list(features.shape)} (2D spatial grid)")
    print(f"  Channels: {in_ch} → {out_ch}")
    print()

print("="*60)
print("Now these features can be processed with ConvNets!")

In [ ]:
# Visualize the reassemble operation

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: 1D tokens
ax = axes[0]
ax.set_title('Step 1: ViT Tokens (1D sequence)', fontsize=12, fontweight='bold')
token_vis = np.random.randn(196, 50)
im = ax.imshow(token_vis, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)
ax.set_xlabel('Feature dimension (768)')
ax.set_ylabel('Token index (0-195)')
ax.text(25, -15, '[B, 196, 768]', ha='center', fontsize=11, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.colorbar(im, ax=ax, fraction=0.046)

# Panel 2: 2D spatial grid
ax = axes[1]
ax.set_title('Step 2: Reshape to 2D grid', fontsize=12, fontweight='bold')
spatial_vis = token_vis[:196, :].reshape(14, 14, 50).mean(axis=2)
im = ax.imshow(spatial_vis, cmap='coolwarm', vmin=-2, vmax=2)
ax.set_xlabel('Width (14 patches)')
ax.set_ylabel('Height (14 patches)')
ax.text(7, -1.5, '[B, 14, 14, 768]', ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
plt.colorbar(im, ax=ax, fraction=0.046)

# Panel 3: After projection
ax = axes[2]
ax.set_title('Step 3: Project to target channels', fontsize=12, fontweight='bold')
projected_vis = spatial_vis * 1.2  # Simulate projection
im = ax.imshow(projected_vis, cmap='viridis')
ax.set_xlabel('Width (14 patches)')
ax.set_ylabel('Height (14 patches)')
ax.text(7, -1.5, '[B, C_out, 14, 14]', ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

print("Reassemble: 1D tokens → 2D spatial grid → Ready for ConvNets!")

## 3. Multi-scale Fusion

### Progressive Refinement Strategy

DPT fuses features from coarse to fine (deep to shallow):

```
┌──────────┐
│ Layer 23 │ (1024 ch, 14×14)  ← Start: deepest features
└─────┬────┘
      │ Upsample 2× + Conv
      ↓
┌──────────┐
│ Layer 17 │ (1024 ch, 14×14)  ← Fuse + Refine
└─────┬────┘
      │ Upsample 2× + Conv  
      ↓
┌──────────┐
│ Layer 11 │ (512 ch, 14×14)   ← Fuse + Refine
└─────┬────┘
      │ Upsample 2× + Conv
      ↓
┌──────────┐
│ Layer 4  │ (256 ch, 14×14)   ← Fuse + Refine
└─────┬────┘
      │ Final Conv
      ↓
┌──────────┐
│  Output  │ (e.g., 2 ch for depth+conf)
└──────────┘
```

### Fusion Operation at Each Stage

```python
fused = upsample(prev_features, scale=2)  # 2× spatial resolution
fused = fused + curr_layer_features       # Element-wise addition
fused = conv_refinement(fused)            # Refine with convolutions
```

In [ ]:
class FusionBlock(nn.Module):
    """
    Single fusion block: upsample → add → refine.
    """
    def __init__(self, channels, upsample_factor=2):
        super().__init__()
        self.upsample_factor = upsample_factor
        
        # Upsampling + refinement
        self.upsample = nn.Upsample(scale_factor=upsample_factor, 
                                   mode='bilinear', align_corners=True)
        
        # Refinement convolutions
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
    def forward(self, prev_features, curr_features):
        """
        Args:
            prev_features: [B, C, H, W] from previous (deeper) layer
            curr_features: [B, C, H', W'] from current layer
        Returns:
            fused: [B, C, H'*2, W'*2] fused and refined features
        """
        # Upsample previous features
        upsampled = self.upsample(prev_features)
        
        # Add current layer features (skip connection)
        fused = upsampled + curr_features
        
        # Refine with convolutions
        fused = self.refine(fused)
        
        return fused


class DPTFusionHead(nn.Module):
    """
    Complete DPT fusion head: multi-scale fusion with progressive refinement.
    """
    def __init__(self, feature_channels=[256, 512, 1024, 1024], output_channels=2):
        super().__init__()
        # feature_channels: [layer4, layer11, layer17, layer23]
        # Reverse for fusion (deep → shallow)
        
        # Project all to common dimension (e.g., 256)
        common_dim = 256
        self.projections = nn.ModuleList([
            nn.Conv2d(ch, common_dim, kernel_size=1) 
            for ch in feature_channels
        ])
        
        # Fusion blocks (3 stages: 23→17, 17→11, 11→4)
        self.fusion1 = FusionBlock(common_dim)  # Layer 23 + 17
        self.fusion2 = FusionBlock(common_dim)  # Result + 11
        self.fusion3 = FusionBlock(common_dim)  # Result + 4
        
        # Final output head
        self.output_conv = nn.Sequential(
            nn.Conv2d(common_dim, common_dim // 2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(common_dim // 2, output_channels, kernel_size=1),
        )
        
    def forward(self, features_dict):
        """
        Args:
            features_dict: dict with keys 'layer4', 'layer11', 'layer17', 'layer23'
                          Each [B, C_in, H, W]
        Returns:
            output: [B, output_channels, H_out, W_out]
        """
        # Project to common dimension
        feat4 = self.projections[0](features_dict['layer4'])
        feat11 = self.projections[1](features_dict['layer11'])
        feat17 = self.projections[2](features_dict['layer17'])
        feat23 = self.projections[3](features_dict['layer23'])
        
        # Progressive fusion (deep → shallow)
        fused = self.fusion1(feat23, feat17)  # Stage 1
        fused = self.fusion2(fused, feat11)   # Stage 2
        fused = self.fusion3(fused, feat4)    # Stage 3
        
        # Final output
        output = self.output_conv(fused)
        
        return output, {'stage1': fused, 'stage2': fused, 'stage3': fused}


# Test the fusion head
print("Testing DPT Fusion Head:\n")

B = 2
H, W = 14, 14  # Patch grid size

# Create synthetic multi-scale features
features_dict = {
    'layer4':  torch.randn(B, 256, H, W),
    'layer11': torch.randn(B, 512, H, W),
    'layer17': torch.randn(B, 1024, H, W),
    'layer23': torch.randn(B, 1024, H, W),
}

# Create and apply fusion head
fusion_head = DPTFusionHead(output_channels=2)  # depth + confidence
output, intermediates = fusion_head(features_dict)

print("Input features:")
for name, feat in features_dict.items():
    print(f"  {name:10s}: {list(feat.shape)}")

print(f"\nOutput: {list(output.shape)}")
print(f"  → {output.shape[1]} channels (e.g., depth + confidence)")
print(f"  → Spatial resolution: {output.shape[2]} × {output.shape[3]}")
print(f"  → Upsampled from {H}×{W} by factor of {output.shape[2]//H}×")

In [ ]:
# Visualize fusion process step by step

# Create more realistic synthetic features
torch.manual_seed(42)
B = 1
H, W = 14, 14

# Simulate features with different characteristics
x = torch.linspace(-1, 1, H).unsqueeze(1).expand(H, W)
y = torch.linspace(-1, 1, W).unsqueeze(0).expand(H, W)

feat23 = (torch.exp(-(x**2 + y**2)/0.8)).unsqueeze(0).unsqueeze(0).expand(B, 256, H, W)
feat17 = (torch.sin(x*3) * torch.cos(y*3) + 1).unsqueeze(0).unsqueeze(0).expand(B, 256, H, W)
feat11 = (torch.sin(x*6) * torch.cos(y*6) + 1).unsqueeze(0).unsqueeze(0).expand(B, 256, H, W)
feat4 = (torch.sin(x*12) * torch.cos(y*12) + torch.randn(H, W)*0.2 + 1).unsqueeze(0).unsqueeze(0).expand(B, 256, H, W)

# Apply fusion blocks
fusion1 = FusionBlock(256)
fusion2 = FusionBlock(256)
fusion3 = FusionBlock(256)

stage1 = fusion1(feat23, feat17)
stage2 = fusion2(stage1, feat11)
stage3 = fusion3(stage2, feat4)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Top row: input features
inputs = [feat23, feat17, feat11, feat4]
titles_top = ['Layer 23 (Deep)\n1024 ch → 256 ch', 
              'Layer 17\n1024 ch → 256 ch',
              'Layer 11\n512 ch → 256 ch', 
              'Layer 4 (Shallow)\n256 ch → 256 ch']

for ax, feat, title in zip(axes[0], inputs, titles_top):
    vis = feat[0, 0].detach().numpy()
    im = ax.imshow(vis, cmap='viridis')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Bottom row: fusion stages
stages = [feat23, stage1, stage2, stage3]
titles_bottom = ['Start: Layer 23\n(14×14)', 
                 'Stage 1: +Layer 17\n(28×28)',
                 'Stage 2: +Layer 11\n(56×56)', 
                 'Stage 3: +Layer 4\n(112×112)']

for ax, feat, title in zip(axes[1], stages, titles_bottom):
    vis = feat[0, 0].detach().numpy()
    # Downsample for visualization consistency
    if vis.shape[0] > 14:
        import scipy.ndimage
        factor = vis.shape[0] / 14
        vis = scipy.ndimage.zoom(vis, 1/factor, order=1)
    im = ax.imshow(vis, cmap='plasma')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Multi-scale Fusion Process: Deep → Shallow', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Fusion Strategy:")
print("  1. Start with deepest features (global context)")
print("  2. Progressively add shallower features (local details)")
print("  3. Upsample at each stage (increase resolution)")
print("  4. Result: High-resolution output with both global and local information!")

## 4. Activation Functions

Different prediction targets need different activation functions:

### 4.1 Depth Prediction: `exp` activation

$$d = \exp(\hat{d})$$

**Why?** Depth must be positive: $d > 0$

### 4.2 Confidence: `expp1` activation

$$c = \exp(\hat{c}) + 1$$

**Why?** Confidence should be $c \geq 1$ (avoids division by zero in loss functions)

### 4.3 Point Coordinates: `inv_log` activation

$$p = \frac{1}{\log(\hat{p} + 2)}$$

**Why?** Maps network output to reasonable 3D coordinate range, smooth gradients

In [ ]:
def exp_activation(x):
    """Exponential activation for depth (ensures positive)."""
    return torch.exp(x)

def expp1_activation(x):
    """Exponential + 1 activation for confidence (ensures >= 1)."""
    return torch.exp(x) + 1

def inv_log_activation(x):
    """Inverse log activation for point coordinates."""
    return 1.0 / torch.log(x + 2)


# Visualize all three activation functions
x = torch.linspace(-3, 3, 1000)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Exp activation
ax = axes[0]
y_exp = exp_activation(x)
ax.plot(x.numpy(), y_exp.numpy(), 'b-', linewidth=2, label='exp(x)')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='y=0 (lower bound)')
ax.fill_between(x.numpy(), 0, y_exp.numpy(), alpha=0.2)
ax.set_xlabel('Network output x', fontsize=11)
ax.set_ylabel('Depth d', fontsize=11)
ax.set_title('Depth: exp(x)\n(Always positive)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 20)

# Expp1 activation
ax = axes[1]
y_expp1 = expp1_activation(x)
ax.plot(x.numpy(), y_expp1.numpy(), 'g-', linewidth=2, label='exp(x) + 1')
ax.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='y=1 (lower bound)')
ax.fill_between(x.numpy(), 1, y_expp1.numpy(), alpha=0.2)
ax.set_xlabel('Network output x', fontsize=11)
ax.set_ylabel('Confidence c', fontsize=11)
ax.set_title('Confidence: exp(x) + 1\n(Always ≥ 1)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 20)

# Inv_log activation
ax = axes[2]
y_inv_log = inv_log_activation(x)
ax.plot(x.numpy(), y_inv_log.numpy(), 'm-', linewidth=2, label='1 / log(x + 2)')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='y=0 (asymptote)')
ax.fill_between(x.numpy(), 0, y_inv_log.numpy(), alpha=0.2)
ax.set_xlabel('Network output x', fontsize=11)
ax.set_ylabel('Coordinate value', fontsize=11)
ax.set_title('Point Coords: 1 / log(x + 2)\n(Smooth, bounded)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(-2, 5)

plt.tight_layout()
plt.show()

print("\nActivation Function Summary:")
print("="*60)
print("1. exp(x):      Depth prediction (must be positive)")
print("2. exp(x) + 1:  Confidence (must be ≥ 1, avoids divide-by-zero)")
print("3. 1/log(x+2):  Point coords (smooth, bounded range)")
print("="*60)

## 5. Depth vs Point Head Comparison

### Two Approaches to 3D Prediction

| Approach | Output | Activation | 3D Recovery |
|----------|--------|------------|-------------|
| **Depth Head** | 2 channels: depth + conf | exp, expp1 | Unproject with K |
| **Point Head** | 4 channels: xyz + conf | inv_log, expp1 | Direct 3D coords |

### Depth Head (Indirect)

```python
# Predict depth per pixel
depth, conf = network(image)  # [B, 2, H, W]
depth = exp(depth)            # Ensure positive
conf = exp(conf) + 1

# Unproject to 3D
points_3d = unproject(depth, K)  # [B, H, W, 3]
```

**Pros**: Respects camera geometry, physically interpretable  
**Cons**: Requires camera intrinsics K

### Point Head (Direct)

```python
# Predict 3D coordinates directly
xyz, conf = network(image)   # [B, 4, H, W]
xyz = inv_log(xyz)           # Smooth activation
conf = exp(conf) + 1

# Already in 3D!
points_3d = xyz.permute(0, 2, 3, 1)  # [B, H, W, 3]
```

**Pros**: No camera calibration needed  
**Cons**: May not respect camera geometry exactly

In [ ]:
class DepthHead(nn.Module):
    """Predicts depth + confidence (2 channels)."""
    def __init__(self, in_channels=256):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 2, kernel_size=1),  # 2 channels: depth, conf
        )
    
    def forward(self, features):
        out = self.head(features)  # [B, 2, H, W]
        depth_logit = out[:, 0:1]  # [B, 1, H, W]
        conf_logit = out[:, 1:2]   # [B, 1, H, W]
        
        depth = torch.exp(depth_logit)       # exp activation
        conf = torch.exp(conf_logit) + 1     # expp1 activation
        
        return depth, conf


class PointHead(nn.Module):
    """Predicts 3D points + confidence (4 channels)."""
    def __init__(self, in_channels=256):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 4, kernel_size=1),  # 4 channels: x, y, z, conf
        )
    
    def forward(self, features):
        out = self.head(features)  # [B, 4, H, W]
        xyz_logit = out[:, 0:3]    # [B, 3, H, W]
        conf_logit = out[:, 3:4]   # [B, 1, H, W]
        
        xyz = 1.0 / torch.log(xyz_logit + 2)  # inv_log activation
        conf = torch.exp(conf_logit) + 1      # expp1 activation
        
        return xyz, conf


def unproject_depth(depth, K):
    """
    Unproject depth map to 3D points.
    
    Args:
        depth: [B, 1, H, W]
        K: [B, 3, 3] camera intrinsics
    Returns:
        points_3d: [B, 3, H, W]
    """
    B, _, H, W = depth.shape
    
    # Pixel grid
    u = torch.arange(W, device=depth.device, dtype=depth.dtype)
    v = torch.arange(H, device=depth.device, dtype=depth.dtype)
    vv, uu = torch.meshgrid(v, u, indexing='ij')
    
    # Back-projection
    fx = K[:, 0, 0].view(B, 1, 1)
    fy = K[:, 1, 1].view(B, 1, 1)
    cx = K[:, 0, 2].view(B, 1, 1)
    cy = K[:, 1, 2].view(B, 1, 1)
    
    X = (uu.unsqueeze(0) - cx) * depth[:, 0] / fx
    Y = (vv.unsqueeze(0) - cy) * depth[:, 0] / fy
    Z = depth[:, 0]
    
    points_3d = torch.stack([X, Y, Z], dim=1)  # [B, 3, H, W]
    return points_3d


# Compare both approaches
print("Comparing Depth Head vs Point Head:\n")
print("="*60)

B, H, W = 2, 56, 56
features = torch.randn(B, 256, H, W)
K = torch.tensor([
    [[30, 0, W/2],
     [0, 30, H/2],
     [0, 0, 1]]
], dtype=torch.float32).expand(B, -1, -1)

# Depth head approach
depth_head = DepthHead()
depth, conf_depth = depth_head(features)
points_from_depth = unproject_depth(depth, K)

print("DEPTH HEAD (Indirect):")
print(f"  Output: depth={list(depth.shape)}, conf={list(conf_depth.shape)}")
print(f"  After unprojection: points_3d={list(points_from_depth.shape)}")
print(f"  Depth range: [{depth.min():.3f}, {depth.max():.3f}]")
print(f"  3D range: X=[{points_from_depth[:,0].min():.2f}, {points_from_depth[:,0].max():.2f}]")
print(f"            Y=[{points_from_depth[:,1].min():.2f}, {points_from_depth[:,1].max():.2f}]")
print(f"            Z=[{points_from_depth[:,2].min():.2f}, {points_from_depth[:,2].max():.2f}]")

# Point head approach
point_head = PointHead()
points_direct, conf_point = point_head(features)

print("\nPOINT HEAD (Direct):")
print(f"  Output: xyz={list(points_direct.shape)}, conf={list(conf_point.shape)}")
print(f"  Already 3D: points_3d={list(points_direct.shape)}")
print(f"  3D range: X=[{points_direct[:,0].min():.2f}, {points_direct[:,0].max():.2f}]")
print(f"            Y=[{points_direct[:,1].min():.2f}, {points_direct[:,1].max():.2f}]")
print(f"            Z=[{points_direct[:,2].min():.2f}, {points_direct[:,2].max():.2f}]")

print("\n" + "="*60)
print("Key Difference:")
print("  Depth Head: Predicts depth (scalar) → unproject with K → 3D")
print("  Point Head: Predicts 3D coordinates directly (no K needed)")
print("="*60)

In [ ]:
# Visualize depth vs point predictions

fig = plt.figure(figsize=(18, 10))

# Row 1: Depth head
ax1 = plt.subplot(2, 3, 1)
im1 = ax1.imshow(depth[0, 0].detach().numpy(), cmap='plasma')
ax1.set_title('Depth Head: Predicted Depth', fontsize=11, fontweight='bold')
plt.colorbar(im1, ax=ax1, fraction=0.046)
ax1.axis('off')

ax2 = plt.subplot(2, 3, 2)
im2 = ax2.imshow(conf_depth[0, 0].detach().numpy(), cmap='viridis')
ax2.set_title('Depth Head: Confidence', fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=ax2, fraction=0.046)
ax2.axis('off')

ax3 = plt.subplot(2, 3, 3, projection='3d')
# Sample points for visualization
stride = 4
pts = points_from_depth[0, :, ::stride, ::stride].detach().reshape(3, -1).numpy()
ax3.scatter(pts[0], pts[2], pts[1], c=pts[2], cmap='plasma', s=1, alpha=0.5)
ax3.set_title('3D Points (unprojected)', fontsize=11, fontweight='bold')
ax3.set_xlabel('X'); ax3.set_ylabel('Z'); ax3.set_zlabel('Y')

# Row 2: Point head
ax4 = plt.subplot(2, 3, 4)
im4 = ax4.imshow(points_direct[0, 2].detach().numpy(), cmap='plasma')  # Z channel
ax4.set_title('Point Head: Z Coordinate', fontsize=11, fontweight='bold')
plt.colorbar(im4, ax=ax4, fraction=0.046)
ax4.axis('off')

ax5 = plt.subplot(2, 3, 5)
im5 = ax5.imshow(conf_point[0, 0].detach().numpy(), cmap='viridis')
ax5.set_title('Point Head: Confidence', fontsize=11, fontweight='bold')
plt.colorbar(im5, ax=ax5, fraction=0.046)
ax5.axis('off')

ax6 = plt.subplot(2, 3, 6, projection='3d')
pts2 = points_direct[0, :, ::stride, ::stride].detach().reshape(3, -1).numpy()
ax6.scatter(pts2[0], pts2[2], pts2[1], c=pts2[2], cmap='plasma', s=1, alpha=0.5)
ax6.set_title('3D Points (direct)', fontsize=11, fontweight='bold')
ax6.set_xlabel('X'); ax6.set_ylabel('Z'); ax6.set_zlabel('Y')

plt.suptitle('Depth Head vs Point Head Comparison', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Both approaches produce 3D points, but via different paths!")

## 6. Confidence Maps

### What is Confidence?

Confidence indicates **how reliable** each prediction is:
- High confidence: Trust this prediction
- Low confidence: Uncertain region (textureless, occluded, etc.)

### Why Predict Confidence?

1. **Filtering**: Remove unreliable points before rendering
2. **Weighting**: Use in loss functions to focus on reliable regions
3. **Fusion**: Combine predictions from multiple views based on confidence

### Using Confidence

```python
# Filter by confidence threshold
mask = confidence > threshold
reliable_points = points_3d[mask]

# Confidence-weighted loss
loss = (confidence * squared_error).mean()

# Multi-view fusion
fused_depth = (conf1 * d1 + conf2 * d2) / (conf1 + conf2)
```

In [ ]:
# Create synthetic confidence maps with realistic patterns

def create_confidence_map(H, W, pattern='textured'):
    """Create synthetic confidence map."""
    x = torch.linspace(-1, 1, W)
    y = torch.linspace(-1, 1, H)
    Y, X = torch.meshgrid(y, x, indexing='ij')
    
    if pattern == 'textured':
        # High confidence in textured regions, low in smooth areas
        texture = torch.sin(X * 10) * torch.cos(Y * 10)
        confidence = torch.exp(-torch.abs(texture)) + 1.5
    elif pattern == 'center':
        # High confidence in center, low at edges
        confidence = torch.exp(-(X**2 + Y**2) / 0.5) * 3 + 1
    elif pattern == 'occlusion':
        # Simulate occlusion boundaries
        confidence = torch.ones_like(X) * 3
        confidence[torch.abs(X - 0.2) < 0.1] = 1.2  # Low confidence at boundary
    else:
        confidence = torch.ones(H, W) * 2
    
    return confidence


# Generate different confidence patterns
H, W = 64, 64
patterns = ['textured', 'center', 'occlusion']
conf_maps = {p: create_confidence_map(H, W, p) for p in patterns}

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Different confidence patterns
titles = ['Texture-based\n(High in textured areas)', 
          'Center-weighted\n(High in center)',
          'Occlusion-aware\n(Low at boundaries)']

for ax, (pattern, title) in zip(axes[0], zip(patterns, titles)):
    im = ax.imshow(conf_maps[pattern].numpy(), cmap='RdYlGn', vmin=1, vmax=4)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Confidence')

# Row 2: Filtered by confidence threshold
threshold = 2.0
for ax, pattern in zip(axes[1], patterns):
    conf = conf_maps[pattern]
    mask = conf > threshold
    filtered = torch.where(mask, conf, torch.zeros_like(conf))
    
    im = ax.imshow(filtered.numpy(), cmap='RdYlGn', vmin=0, vmax=4)
    ax.set_title(f'Filtered (conf > {threshold})\nRetained: {mask.float().mean()*100:.1f}%', 
                fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Confidence')

plt.suptitle('Confidence Maps: Patterns and Filtering', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nConfidence Map Usage:")
print("  - Filter unreliable predictions before rendering")
print("  - Weight loss function to focus on reliable regions")
print("  - Fuse multi-view predictions based on confidence")

In [ ]:
# Demonstrate confidence-weighted point cloud fusion

def confidence_weighted_fusion(depth1, conf1, depth2, conf2):
    """
    Fuse two depth predictions using confidence weighting.
    """
    fused_depth = (conf1 * depth1 + conf2 * depth2) / (conf1 + conf2 + 1e-8)
    fused_conf = conf1 + conf2  # Combined confidence
    return fused_depth, fused_conf


# Create two synthetic depth maps with different noise levels
x = torch.linspace(-1, 1, W)
y = torch.linspace(-1, 1, H)
Y, X = torch.meshgrid(y, x, indexing='ij')

# Ground truth depth surface
depth_gt = 5.0 + torch.sin(X * 3) * torch.cos(Y * 3)

# View 1: Noisy in left half
noise1 = torch.randn_like(X) * 0.5
noise1[:, W//2:] *= 0.1  # Less noise in right half
depth1 = depth_gt + noise1
conf1 = torch.ones_like(X) * 2
conf1[:, :W//2] = 1.2  # Lower confidence in noisy region

# View 2: Noisy in right half
noise2 = torch.randn_like(X) * 0.5
noise2[:, :W//2] *= 0.1  # Less noise in left half
depth2 = depth_gt + noise2
conf2 = torch.ones_like(X) * 2
conf2[:, W//2:] = 1.2  # Lower confidence in noisy region

# Fuse
depth_fused, conf_fused = confidence_weighted_fusion(depth1, conf1, depth2, conf2)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# View 1
im = axes[0, 0].imshow(depth1.numpy(), cmap='plasma', vmin=3, vmax=7)
axes[0, 0].set_title('View 1: Depth\n(Noisy left)', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')
plt.colorbar(im, ax=axes[0, 0], fraction=0.046)

im = axes[1, 0].imshow(conf1.numpy(), cmap='RdYlGn', vmin=1, vmax=2.5)
axes[1, 0].set_title('View 1: Confidence', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im, ax=axes[1, 0], fraction=0.046)

# View 2
im = axes[0, 1].imshow(depth2.numpy(), cmap='plasma', vmin=3, vmax=7)
axes[0, 1].set_title('View 2: Depth\n(Noisy right)', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im, ax=axes[0, 1], fraction=0.046)

im = axes[1, 1].imshow(conf2.numpy(), cmap='RdYlGn', vmin=1, vmax=2.5)
axes[1, 1].set_title('View 2: Confidence', fontsize=11, fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im, ax=axes[1, 1], fraction=0.046)

# Fused
im = axes[0, 2].imshow(depth_fused.numpy(), cmap='plasma', vmin=3, vmax=7)
axes[0, 2].set_title('Fused: Depth\n(Best of both)', fontsize=11, fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046)

im = axes[1, 2].imshow(conf_fused.numpy(), cmap='RdYlGn', vmin=1, vmax=4.5)
axes[1, 2].set_title('Fused: Confidence', fontsize=11, fontweight='bold')
axes[1, 2].axis('off')
plt.colorbar(im, ax=axes[1, 2], fraction=0.046)

# Error comparison
error1 = torch.abs(depth1 - depth_gt)
error2 = torch.abs(depth2 - depth_gt)
error_fused = torch.abs(depth_fused - depth_gt)

im = axes[0, 3].imshow(error_fused.numpy(), cmap='Reds', vmin=0, vmax=0.5)
axes[0, 3].set_title(f'Fused Error\nMAE: {error_fused.mean():.3f}', 
                    fontsize=11, fontweight='bold')
axes[0, 3].axis('off')
plt.colorbar(im, ax=axes[0, 3], fraction=0.046)

# Error reduction
error_reduction = (error1.mean() + error2.mean()) / 2 - error_fused.mean()
axes[1, 3].bar(['View 1', 'View 2', 'Fused'], 
              [error1.mean(), error2.mean(), error_fused.mean()],
              color=['coral', 'skyblue', 'lightgreen'])
axes[1, 3].set_ylabel('Mean Absolute Error')
axes[1, 3].set_title(f'Error Comparison\nReduction: {error_reduction:.3f}', 
                    fontsize=11, fontweight='bold')
axes[1, 3].grid(True, alpha=0.3)

plt.suptitle('Confidence-weighted Multi-view Fusion', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"\nFusion Results:")
print(f"  View 1 MAE: {error1.mean():.4f}")
print(f"  View 2 MAE: {error2.mean():.4f}")
print(f"  Fused MAE:  {error_fused.mean():.4f}")
print(f"  Improvement: {error_reduction/((error1.mean()+error2.mean())/2)*100:.1f}%")
print("\nConfidence weighting automatically chooses the most reliable prediction!")

## Summary

```
╔═══════════════════════════════════════════════════════════════════╗
║      Notebook 04 Summary: DPT Head Multi-scale Fusion           ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. MULTI-SCALE FEATURES                                         ║
║     - Shallow: local details (edges, textures)                   ║
║     - Deep: global semantics (context, structure)                ║
║     - DPT uses 4 layers: 4, 11, 17, 23                          ║
║                                                                  ║
║  2. REASSEMBLE OPERATION                                         ║
║     - [B*S, P, C] → reshape → [B*S, H_p, W_p, C]               ║
║     - Project to target channels via Conv1x1                     ║
║     - Restores 2D spatial structure for ConvNets                 ║
║                                                                  ║
║  3. PROGRESSIVE FUSION                                           ║
║     - Start from deepest features (layer 23)                     ║
║     - Upsample + fuse with shallower layers                      ║
║     - Each stage: upsample → add → refine                        ║
║                                                                  ║
║  4. ACTIVATION FUNCTIONS                                         ║
║     - exp: depth (must be positive)                              ║
║     - exp + 1: confidence (must be ≥ 1)                         ║
║     - 1/log: point coordinates (smooth, bounded)                 ║
║                                                                  ║
║  5. DEPTH VS POINT HEADS                                         ║
║     - Depth head: 2 ch (depth + conf) → unproject with K        ║
║     - Point head: 4 ch (xyz + conf) → direct 3D                 ║
║                                                                  ║
║  6. CONFIDENCE MAPS                                              ║
║     - Filter unreliable predictions                              ║
║     - Weight loss functions                                      ║
║     - Fuse multi-view predictions                                ║
║                                                                  ║
╚═══════════════════════════════════════════════════════════════════╝
```

## What's Next?

**[05_training_loss.ipynb](./05_training_loss.ipynb)** - Training objectives and loss functions

---

## References

1. DPT (Vision Transformers for Dense Prediction): https://arxiv.org/abs/2103.13413
2. MVSplat (Feed-forward 3D Gaussian Splatting): https://arxiv.org/abs/2403.14627
3. DepthSplat (Connecting Depth Prediction and 3DGS): https://arxiv.org/abs/2410.13862